# Ablation Study



In [ ]:
import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import cross_validate

In [ ]:
import pandas as pd
import numpy as np
import sys
import os

sys.path.append(os.path.abspath(".."))

from src.feature_engineering import (
    sort_and_reset,
    generate_rolling_features,
    merge_external_context
)

In [ ]:
import importlib
import src.feature_engineering as fe

importlib.reload(fe)

In [ ]:
df = pd.read_csv("../data/ai4i2020.csv")

df = sort_and_reset(df)
df = generate_rolling_features(df)
df = merge_external_context(df)

print(df.shape)
df.head()

In [ ]:
print(df.columns.tolist())

In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

df["Type_enc"] = le.fit_transform(df["Type"])

print(df[["Type", "Type_enc"]].head())

In [ ]:
print("Type_enc" in df.columns)

# Without External Features

In [ ]:
base_features = [
    "Type_enc",

    "Air temperature [K]_rolling_mean",
    "Air temperature [K]_rolling_std",
    "Air temperature [K]_rolling_var",

    "Process temperature [K]_rolling_mean",
    "Process temperature [K]_rolling_std",
    "Process temperature [K]_rolling_var",

    "Rotational speed [rpm]_rolling_mean",
    "Rotational speed [rpm]_rolling_std",
    "Rotational speed [rpm]_rolling_var",

    "Torque [Nm]_rolling_mean",
    "Torque [Nm]_rolling_std",
    "Torque [Nm]_rolling_var",

    "Tool wear [min]_rolling_mean",
    "Tool wear [min]_rolling_std",
    "Tool wear [min]_rolling_var"
]

print("Total Base Features:", len(base_features))

In [ ]:
# Input Features
X_base = df[base_features]

# Target Column
y = df["Machine failure"]

print("Shape of X_base:", X_base.shape)
print("Shape of y:", y.shape)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold

# Random Forest Model
rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

# 5-Fold Cross Validation
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

print("Random Forest Model Created Successfully")
print("Cross Validation:", cv)

In [ ]:
from sklearn.model_selection import cross_validate

base_scores = cross_validate(
    estimator=rf,
    X=X_base,
    y=y,
    cv=cv,
    scoring=[
        "f1_macro",
        "precision_macro",
        "recall_macro"
    ]
)

print("Base Model Evaluation Completed")

In [ ]:
base_f1 = base_scores["test_f1_macro"].mean()
base_precision = base_scores["test_precision_macro"].mean()
base_recall = base_scores["test_recall_macro"].mean()

print("=== Base Model Results ===")
print(f"Macro F1 : {base_f1:.4f}")
print(f"Precision: {base_precision:.4f}")
print(f"Recall   : {base_recall:.4f}")

## With External Features


In [ ]:
ext_features = base_features + [
    "ambient_temp_C",
    "factory_load_pct",
    "humidity_pct"
]

print("Total Extended Features:", len(ext_features))

In [ ]:
X_ext = df[ext_features]

print("Shape of X_ext:", X_ext.shape)

In [ ]:
ext_scores = cross_validate(
    estimator=rf,
    X=X_ext,
    y=y,
    cv=cv,
    scoring=[
        "f1_macro",
        "precision_macro",
        "recall_macro"
    ]
)

print("Extended Model Evaluation Completed")

In [ ]:
ext_f1 = ext_scores["test_f1_macro"].mean()
ext_precision = ext_scores["test_precision_macro"].mean()
ext_recall = ext_scores["test_recall_macro"].mean()

print("=== Extended Model Results ===")
print(f"Macro F1 : {ext_f1:.4f}")
print(f"Precision: {ext_precision:.4f}")
print(f"Recall   : {ext_recall:.4f}")

## F1 Improvement


In [ ]:
# Calculate Percentage F1 Improvement
f1_improvement = ((ext_f1 - base_f1) / base_f1) * 100

print("========== F1 Improvement ==========")
print(f"Base Macro F1     : {base_f1:.4f}")
print(f"Extended Macro F1 : {ext_f1:.4f}")
print(f"F1 Improvement    : {f1_improvement:.2f}%")

In [ ]:
comparison = pd.DataFrame({

    "Feature Set": [
        "Without External Features",
        "With External Features"
    ],

    "Macro F1": [
        round(base_f1, 4),
        round(ext_f1, 4)
    ],

    "Precision": [
        round(base_precision, 4),
        round(ext_precision, 4)
    ],

    "Recall": [
        round(base_recall, 4),
        round(ext_recall, 4)
    ]

})

comparison

# Without External Features Rolling

In [192]:
df_roll = generate_rolling_features(df)

print(df_roll.shape)

Output Shape : (9992, 33)
Rolling Features Added : 15
(9992, 33)


In [193]:
rolling_cols = [col for col in df_roll.columns if "rolling" in col]

print(len(rolling_cols))
print(rolling_cols)

15
['Air temperature [K]_rolling_mean', 'Process temperature [K]_rolling_mean', 'Rotational speed [rpm]_rolling_mean', 'Torque [Nm]_rolling_mean', 'Tool wear [min]_rolling_mean', 'Air temperature [K]_rolling_std', 'Process temperature [K]_rolling_std', 'Rotational speed [rpm]_rolling_std', 'Torque [Nm]_rolling_std', 'Tool wear [min]_rolling_std', 'Air temperature [K]_rolling_var', 'Process temperature [K]_rolling_var', 'Rotational speed [rpm]_rolling_var', 'Torque [Nm]_rolling_var', 'Tool wear [min]_rolling_var']


In [194]:
base_features = [
    col for col in df_roll.columns
    if col not in [
        'Machine failure',
        'UDI',
        'Product ID',
        'Type',
        'TWF',
        'HDF',
        'PWF',
        'OSF',
        'RNF'
    ]
]

print(len(base_features))


24


In [195]:
X = df_roll[base_features]

y = df_roll['Machine failure']

print(X.shape)
print(y.shape)

(9992, 24)
(9992,)


In [196]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold

rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [197]:
from sklearn.model_selection import cross_validate

scores = cross_validate(
    rf,
    X,
    y,
    cv=cv,
    scoring='f1_macro'
)

macro_f1 = scores['test_score'].mean()

print("Macro F1 Score =", round(macro_f1,4))

Macro F1 Score = 0.7897


## Without External Features

Model Used:
RandomForestClassifier

Validation Method:
5-Fold StratifiedKFold Cross Validation

Feature Set:
- Type_enc
- Original IoT Sensor Features
- Rolling Mean Features
- Rolling Standard Deviation Features
- Rolling Variance Features

Result:
Macro F1 Score = 0.8104

Conclusion:
The baseline Random Forest model achieved a Macro F1 Score of 0.8104 using only internal IoT telemetry and rolling statistical features. This result will be used as a benchmark before adding external contextual features.

## With External Features rolling

In [198]:
import sys
import os

sys.path.append(os.path.abspath(".."))

In [199]:
from src.feature_sets import ext_features

print(len(ext_features))
print(ext_features)

19
['Type_enc', 'Air temperature [K]_rolling_mean', 'Process temperature [K]_rolling_mean', 'Rotational speed [rpm]_rolling_mean', 'Torque [Nm]_rolling_mean', 'Tool wear [min]_rolling_mean', 'Air temperature [K]_rolling_std', 'Process temperature [K]_rolling_std', 'Rotational speed [rpm]_rolling_std', 'Torque [Nm]_rolling_std', 'Tool wear [min]_rolling_std', 'Air temperature [K]_rolling_var', 'Process temperature [K]_rolling_var', 'Rotational speed [rpm]_rolling_var', 'Torque [Nm]_rolling_var', 'Tool wear [min]_rolling_var', 'ambient_temp_C', 'factory_load_pct', 'humidity_pct']


In [200]:
df_roll = generate_rolling_features(df)

print(df_roll.shape)

Output Shape : (9992, 33)
Rolling Features Added : 15
(9992, 33)


In [201]:
print(df_roll.head())

   UDI Product ID Type  Air temperature [K]  Process temperature [K]  \
0    9     M14868    M                298.3                    308.7   
1   10     M14869    M                298.5                    309.0   
2   11     H29424    H                298.4                    308.9   
3   12     H29425    H                298.6                    309.1   
4   13     M14872    M                298.6                    309.1   

   Rotational speed [rpm]  Torque [Nm]  Tool wear [min]  Machine failure  TWF  \
0                    1667         28.6               18                0    0   
1                    1741         28.0               21                0    0   
2                    1782         23.9               24                0    0   
3                    1423         44.3               29                0    0   
4                    1339         51.1               34                0    0   

   ...  Tool wear [min]_rolling_std  Air temperature [K]_rolling_var  \
0  ...  

In [202]:
print(df_roll.columns.tolist())

['UDI', 'Product ID', 'Type', 'Air temperature [K]', 'Process temperature [K]', 'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]', 'Machine failure', 'TWF', 'HDF', 'PWF', 'OSF', 'RNF', 'Air temperature [K]_rolling_mean', 'Process temperature [K]_rolling_mean', 'Rotational speed [rpm]_rolling_mean', 'Torque [Nm]_rolling_mean', 'Tool wear [min]_rolling_mean', 'Air temperature [K]_rolling_std', 'Process temperature [K]_rolling_std', 'Rotational speed [rpm]_rolling_std', 'Torque [Nm]_rolling_std', 'Tool wear [min]_rolling_std', 'Air temperature [K]_rolling_var', 'Process temperature [K]_rolling_var', 'Rotational speed [rpm]_rolling_var', 'Torque [Nm]_rolling_var', 'Tool wear [min]_rolling_var', 'ambient_temp_C', 'factory_load_pct', 'humidity_pct', 'Type_enc']


In [203]:
print(df.columns.tolist())

['UDI', 'Product ID', 'Type', 'Air temperature [K]', 'Process temperature [K]', 'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]', 'Machine failure', 'TWF', 'HDF', 'PWF', 'OSF', 'RNF', 'Air temperature [K]_rolling_mean', 'Process temperature [K]_rolling_mean', 'Rotational speed [rpm]_rolling_mean', 'Torque [Nm]_rolling_mean', 'Tool wear [min]_rolling_mean', 'Air temperature [K]_rolling_std', 'Process temperature [K]_rolling_std', 'Rotational speed [rpm]_rolling_std', 'Torque [Nm]_rolling_std', 'Tool wear [min]_rolling_std', 'Air temperature [K]_rolling_var', 'Process temperature [K]_rolling_var', 'Rotational speed [rpm]_rolling_var', 'Torque [Nm]_rolling_var', 'Tool wear [min]_rolling_var', 'ambient_temp_C', 'factory_load_pct', 'humidity_pct', 'Type_enc']


In [204]:
print(df.shape)

(9996, 33)


In [205]:
external_cols = [
    "ambient_temp_C",
    "factory_load_pct",
    "humidity_pct"
]

for col in external_cols:
    print(col, "->", col in df.columns)

ambient_temp_C -> True
factory_load_pct -> True
humidity_pct -> True


In [206]:
import numpy as np

np.random.seed(42)

df_roll["ambient_temp_C"] = np.random.normal(
    loc=28,
    scale=5,
    size=len(df_roll)
)

df_roll["factory_load_pct"] = np.random.uniform(
    50,
    100,
    size=len(df_roll)
)

df_roll["humidity_pct"] = np.random.normal(
    loc=60,
    scale=10,
    size=len(df_roll)
)

In [207]:
print(
    df_roll[
        [
            "ambient_temp_C",
            "factory_load_pct",
            "humidity_pct"
        ]
    ].head()
)

   ambient_temp_C  factory_load_pct  humidity_pct
0       30.483571         63.311414     71.687689
1       27.308678         80.164576     60.552304
2       31.238443         83.445005     56.516916
3       35.615149         82.751235     61.923689
4       26.829233         72.292482     53.418912


In [208]:
from src.feature_sets import ext_features

X_ext = df_roll[ext_features]

y_ext = df_roll["Machine failure"]

print(X_ext.shape)
print(y_ext.shape)

(9992, 19)
(9992,)


In [209]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

print(rf)

RandomForestClassifier(random_state=42)


In [210]:
from sklearn.model_selection import StratifiedKFold

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

print(cv)

StratifiedKFold(n_splits=5, random_state=42, shuffle=True)


In [211]:
from sklearn.model_selection import cross_validate

scores_ext = cross_validate(
    rf,
    X_ext,
    y_ext,
    cv=cv,
    scoring="f1_macro"
)

ext_macro_f1 = scores_ext["test_score"].mean()

print("Fold Scores:", scores_ext["test_score"])
print("With External Features Macro F1 =", round(ext_macro_f1, 4))

Fold Scores: [0.49083036 0.49096002 0.5052947  0.51810301 0.51911315]
With External Features Macro F1 = 0.5049


In [212]:
print(ext_macro_f1)

0.5048602477408541


In [213]:
print("Without External Features =", round(macro_f1, 4))
print("With External Features    =", round(ext_macro_f1, 4))

Without External Features = 0.7897
With External Features    = 0.5049


## Conclusion

Without External Features:
Macro F1 = 0.8008

With External Features:
Macro F1 = 0.5027

The addition of simulated external context features did not improve model performance in this experiment. The Random Forest model achieved a lower Macro F1 score when external features were included.